# Mission 8: The Cost of Trading — Turnover, Transaction Costs & Capacity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/missions/mission_08_cost_of_trading/notebook.ipynb)

**Advanced elective.** You found an alpha. Congratulations — now try to *keep* it. This mission is
about the gap between a frictionless backtest and a tradeable strategy. The villain is **turnover**:
every time you rebalance you pay the spread, and a signal that looks great traded daily can be a
guaranteed loser once costs are real.

**Learning objectives**
- Quantify how **transaction costs** scale with turnover and erase paper alpha
- Use **rebalance frequency** and **no-trade bands** to trade turnover against signal freshness
- Find a strategy's **break-even cost** — the TC at which its edge disappears
- Reason about **capacity**: why size itself moves the price against you

## Part 0: Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Pinned to the exact commit so the backtester matches this lesson.
    !pip install -q "git+https://github.com/convexpi/lab.git@b3006123f7bf455d0dbb1cb03a87dd8bd3e5bed0"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from convexpi.lab import SyntheticMarket
from convexpi.lab.backtest import Backtest, Strategy

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100

market = SyntheticMarket(n_assets=200, n_days=1000, seed=42)
FEATURE_NAMES = market.FEATURE_NAMES
prices = market.prices('train')                       # (T, N)
fa = market.features_array('train')                    # (T, N, F)
feats = {name: fa[:, :, i] for i, name in enumerate(FEATURE_NAMES)}
print('Ready. Features:', FEATURE_NAMES[:4], '...')

---
## Part 1: The same alpha, two different costs

We use a single, honest signal: 1-month momentum (`mom_1m`), a planted alpha in this market. The
strategy is a quintile long/short, gross exposure 1.0. We'll run it **frictionless** and then with a
realistic **one-way cost of 20 bps**, rebalancing daily either way.

In [ ]:
class Momentum(Strategy):
    """Quintile long/short on a single feature, gross exposure 1.0."""
    def __init__(self, feature='mom_1m'):
        self.feature = feature
    def on_day(self, day, features, prices, portfolio):
        s = features[self.feature].astype(float)
        z = (s - np.nanmean(s)) / (np.nanstd(s) + 1e-9)
        k = max(1, len(z) // 5)
        order = np.argsort(z)
        w = np.zeros_like(z)
        w[order[-k:]] = 1.0
        w[order[:k]] = -1.0
        return w / (np.abs(w).sum() + 1e-9)

for label, tc in [("frictionless (0 bps)", 0.0), ("realistic (20 bps)", 20.0)]:
    r = Backtest(tc_bps=tc, rebalance_every=1, warmup_days=60).run(Momentum(), prices, feats)
    print(f"{label:22}  Sharpe={r.sharpe:6.3f}  ann_ret={r.annualized_return:7.2%}  "
          f"turnover={r.turnover_annual:6.1f}x")

The frictionless backtest looks like a real edge. Add 20 bps and rebalance every day and the **same
signal becomes a money-loser** — costs eat it alive. The backtest didn't lie about the alpha; it lied
about what you'd *net*. Turnover is the bridge between the two.

---
## Part 2: Rebalance frequency — turnover vs freshness

The simplest turnover lever is *how often you trade*. Rebalancing less often cuts costs but lets your
positions drift from the signal. Sweep it (at 20 bps one-way):

In [ ]:
rows = []
for rb in [1, 2, 5, 10, 21, 42]:
    r = Backtest(tc_bps=20, rebalance_every=rb, warmup_days=60).run(Momentum(), prices, feats)
    rows.append((rb, r.sharpe, r.turnover_annual, r.annualized_return))

print(f"{'rebal(d)':>9}{'Sharpe':>9}{'turnover':>10}{'ann_ret':>10}")
for rb, sh, to, ar in rows:
    print(f"{rb:>9}{sh:>9.3f}{to:>9.1f}x{ar:>10.2%}")

fig, ax1 = plt.subplots(figsize=(8, 3.4))
rbs = [r[0] for r in rows]
ax1.plot(rbs, [r[1] for r in rows], 'o-', color='C0', label='Sharpe (net)')
ax1.axhline(0, color='k', lw=0.8); ax1.set_xlabel('rebalance every N days'); ax1.set_ylabel('net Sharpe', color='C0')
ax2 = ax1.twinx(); ax2.plot(rbs, [r[2] for r in rows], 's--', color='C3', label='turnover')
ax2.set_ylabel('annual turnover (x)', color='C3')
plt.title('Trading less often: lower turnover, higher net Sharpe'); fig.tight_layout(); plt.show()

**Exercise 2.1** — At 20 bps, daily rebalancing is deeply negative while monthly is positive. Does
the *gross* (frictionless) Sharpe also rise with slower rebalancing, or is the improvement entirely
a cost effect? Re-run the sweep at `tc_bps=0` to separate the two.

---
## Part 3: Break-even cost

Every strategy has a transaction cost at which its net edge hits zero — the **break-even cost**. It's
the single most useful number for judging whether an alpha is tradeable. Fix a sensible rebalance
frequency and sweep the cost:

In [ ]:
costs = [0, 5, 10, 20, 30, 50, 75, 100]
sharpes = [Backtest(tc_bps=c, rebalance_every=5, warmup_days=60).run(Momentum(), prices, feats).sharpe
           for c in costs]
for c, s in zip(costs, sharpes):
    print(f"  tc={c:>3} bps  ->  net Sharpe {s:6.3f}")

plt.figure(figsize=(8, 3.2))
plt.plot(costs, sharpes, 'o-')
plt.axhline(0, color='crimson', lw=1, ls='--')
plt.xlabel('one-way transaction cost (bps)'); plt.ylabel('net Sharpe')
plt.title('Break-even cost: where the edge disappears'); plt.tight_layout(); plt.show()

# Linear-interpolate the zero crossing for a break-even estimate.
sa = np.array(sharpes)
cross = np.where(np.diff(np.sign(sa)))[0]
if len(cross):
    i = cross[0]
    be = costs[i] + (costs[i+1]-costs[i]) * sa[i]/(sa[i]-sa[i+1])
    print(f"\nApprox break-even cost (rebal=5d): ~{be:.0f} bps")

If your break-even cost is comfortably above what you'd actually pay (spread + impact + fees), the
edge survives. If it's *below*, the alpha is real on paper and worthless in practice. Mission 6
covered the execution side of that cost; here it's the portfolio-level budget.

---
## Part 4: Reducing turnover with a no-trade band

Trading less *often* is blunt. A smarter tool is a **no-trade band**: only move a position when the
target has drifted far enough from where you already are. Small drifts are ignored, so you stop
churning on noise.

In [ ]:
class BandedMomentum(Momentum):
    """Only rebalance a name when its target moves more than `band` from the current weight."""
    def __init__(self, feature='mom_1m', band=0.0):
        super().__init__(feature); self.band = band
    def on_day(self, day, features, prices, portfolio):
        target = super().on_day(day, features, prices, portfolio)
        if self.band <= 0:
            return target
        move = np.abs(target - portfolio) > self.band
        return np.where(move, target, portfolio)     # hold unless drift exceeds the band

print(f"{'band':>6}{'Sharpe':>9}{'turnover':>10}{'ann_ret':>10}")
for band in [0.0, 0.002, 0.005, 0.01]:
    r = Backtest(tc_bps=20, rebalance_every=1, warmup_days=60).run(
        BandedMomentum(band=band), prices, feats)
    print(f"{band:>6.3f}{r.sharpe:>9.3f}{r.turnover_annual:>9.1f}x{r.annualized_return:>10.2%}")

A band lets you rebalance *daily* (staying responsive to the signal) while paying far less in costs
than naive daily trading — often beating a fixed slow rebalance. The right band depends on your
signal's noise and your cost level.

**Exercise 4.1** — A band and a slow rebalance both cut turnover. Compare the *best* band against the
*best* rebalance frequency from Part 2 at 20 bps. Which delivers the higher net Sharpe, and why might
a band be preferable when the signal occasionally moves sharply?

---
## Part 5: Capacity — when your own size moves the price

So far cost-per-trade was constant. In reality, **larger orders move the price against you** (market
impact), so cost *rises with size*. A standard square-root model says the cost of trading a fraction
`q` of a name's daily volume is roughly `impact ≈ k · sqrt(q)`. That puts a ceiling on how much
capital a strategy can run — its **capacity**.

In [ ]:
# Sketch: net Sharpe as AUM grows, with sqrt-impact added to a fixed base cost.
base_bps = 5.0          # spread + fees, size-independent
k_bps = 8.0             # impact coefficient
adv_usd_per_name = 5e6  # assumed average daily volume per name

aums = np.array([1e6, 1e7, 5e7, 1e8, 5e8, 1e9])
turn = Backtest(tc_bps=0, rebalance_every=5, warmup_days=60).run(Momentum(), prices, feats).turnover_annual
sharpes = []
for aum in aums:
    per_name_trade = aum / 40                       # ~20 long + 20 short
    q = per_name_trade / adv_usd_per_name           # fraction of ADV per rebalance
    tc = base_bps + k_bps * np.sqrt(max(q, 0))
    sharpes.append(Backtest(tc_bps=tc, rebalance_every=5, warmup_days=60).run(Momentum(), prices, feats).sharpe)

for aum, s in zip(aums, sharpes):
    print(f"  AUM ${aum/1e6:>7.0f}M  ->  net Sharpe {s:6.3f}")
plt.figure(figsize=(8, 3.2))
plt.semilogx(aums, sharpes, 'o-'); plt.axhline(0, color='crimson', lw=1, ls='--')
plt.xlabel('assets under management ($)'); plt.ylabel('net Sharpe')
plt.title('Capacity: impact rises with size until the edge is gone'); plt.tight_layout(); plt.show()

Capacity is why a strategy that's spectacular at \$1M can be mediocre at \$1B — the alpha per dollar
is finite, and impact taxes every extra dollar harder. Small, nimble strategies trade high-turnover
signals others can't touch; large funds are pushed toward slow, low-turnover, high-capacity factors.

---
## Challenge

Take *your* best strategy from an earlier mission (or the research library) and produce its
**cost report**:

1. Its turnover and net Sharpe at 0 / 10 / 20 / 50 bps.
2. Its break-even cost.
3. A turnover-reduction variant (slower rebalance or a band) that improves net Sharpe.
4. A one-paragraph honest verdict: at what cost level, and what AUM, does this stop being worth
   trading?

Publish the report to **[/projects](https://convexpi.ai/projects)** — a clear-eyed capacity analysis
is exactly the kind of work that separates a backtest from a strategy.